# SDK callback → Python Polars 性能复核

本 notebook 只读取已由 fail-closed harness 验证并发布 `summary.json` 完成标记的正式 campaign。吞吐与延迟来自不同的新进程，不能把下面的延迟称为 ‘500k/s 下的延迟’。

In [ ]:
from pathlib import Path
import json, math, statistics
import polars as pl

ROOT = Path.cwd()
if not (ROOT / 'artifacts').is_dir():
    ROOT = ROOT.parent

def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

fast = load_json('artifacts/perf-20260803-fast-current-w0/summary.json')
partial = load_json('artifacts/perf-20260803-partial-current-w0/summary.json')
certified = load_json('artifacts/perf-20260803-from-open-certified-w0/summary.json')
callback_latency = load_json('artifacts/perf-20260803-sdk-python-w0/latency_runs.json')
recovery = load_json('artifacts/perf-20260803-online-recovery-w0-v2/summary.json')
fast_runs = load_json('artifacts/perf-20260803-fast-current-w0/throughput_runs.json')
partial_runs = load_json('artifacts/perf-20260803-partial-current-w0/throughput_runs.json')
certified_runs = load_json('artifacts/perf-20260803-from-open-certified-w0/throughput_runs.json')

throughput_hashes = {x['provenance']['binary_sha256'] for x in (fast, partial, certified)}
assert len(throughput_hashes) == 1, throughput_hashes
throughput_hashes

In [ ]:
for topology, runs in [('fast', fast_runs), ('partial', partial_runs), ('certified', certified_runs)]:
    assert len(runs) == 9
    for row in runs:
        planned = int(row['planned_callbacks'])
        assert all(int(row[key]) == planned for key in ('invoked_callbacks', 'accepted', 'decoded', 'applied', 'store_appended', 'history_scan_records', 'history_unique_ingress'))
        assert sum(int(row[f'tuple{i}_offered']) for i in range(5)) == planned
        assert int(row['history_duplicate_ingress']) == int(row['history_out_of_range_ingress']) == int(row['history_invalid_source_slots']) == 0
        if topology == 'certified':
            eligible = 3 * planned // 5
            assert int(row['certified_enqueued_observations']) == eligible
            assert int(row['certified_enqueued_applied']) == eligible
            assert int(row['certified_processed']) == 2 * eligible
            assert int(row['certified_event_count']) == 4 * planned // 5
            assert int(row['certified_dropped']) == int(row['certified_global_frozen']) == 0
        elif topology == 'partial':
            assert int(row['partial_event_queue_high_water']) == 262144
            assert int(row['partial_event_dropped']) == 1
'independent closure audit: 27/27 runs PASS'

In [ ]:
rows = []
for topology, report in [('from_open_FAST', fast), ('partial_Event', partial), ('from_open_Certified_Event', certified)]:
    for row in report['throughput_summary']:
        rows.append({
            'topology': topology,
            'target_message_s': row['target_rps'],
            'offered_p50': row['offered_rps_p50'],
            'history_ready_p50': row['history_ready_rps_p50'],
            'full_path_ready_p50': row['full_path_ready_rps_p50'],
            'lossless_runs': row['fast_history_lossless_runs'],
            'full_path_pass_runs': row['full_path_pass_runs'],
            'event_backlog_end_max': row['event_backlog_at_offer_end_max'],
        })
throughput_table = pl.DataFrame(rows).sort(['target_message_s', 'topology'])
throughput_table

In [ ]:
certified_500 = []
for row in certified_runs:
    if int(row['target_rps']) == 500_000:
        certified_500.append({
            'repeat': row['repeat'],
            'q25': int(row['event_backlog_q25']),
            'q50': int(row['event_backlog_q50']),
            'q75': int(row['event_backlog_q75']),
            'q100': int(row['event_backlog_q100']),
            'offer_end': int(row['event_backlog_before_drain']),
            'full_path_message_s': float(row['full_path_ready_rps']),
            'derived_events': int(row['certified_event_count']),
        })
certified_500_table = pl.DataFrame(certified_500)
assert certified_500_table['offer_end'].min() > 1024
certified_500_table

In [ ]:
partial_overflow = []
for row in partial_runs:
    enqueued = int(row['partial_event_enqueued'])
    rate = int(row['target_rps'])
    partial_overflow.append({
        'target_message_s': rate,
        'repeat': row['repeat'],
        'successful_handoffs_before_first_drop': enqueued,
        'observed_native_before_freeze': int(row['partial_event_observed_native']),
        'applied_before_freeze': int(row['partial_event_applied_records']),
        'queue_high_water': int(row['partial_event_queue_high_water']),
        'dropped_handoffs': int(row['partial_event_dropped']),
    })
partial_overflow_table = pl.DataFrame(partial_overflow)
assert partial_overflow_table['queue_high_water'].unique().to_list() == [262144]
assert partial_overflow_table['dropped_handoffs'].unique().to_list() == [1]
partial_overflow_table

In [ ]:
def r7(values, probability):
    ordered = sorted(float(x) for x in values)
    position = (len(ordered) - 1) * probability
    lo, hi = math.floor(position), math.ceil(position)
    return ordered[lo] if lo == hi else ordered[lo] + (ordered[hi] - ordered[lo]) * (position - lo)

latency_rows = []
for scenario in ('from_open', 'live_partial_no_recovery'):
    selected = [r for r in callback_latency if r['scenario'] == scenario]
    for workload, metric in [('raw_4096x55', 'batch_strict_last_callback_to_polars_ns'), ('derived_order_6rows', 'order_strict_last_callback_to_polars_ns')]:
        values = [r[metric] / 1e6 for r in selected]
        latency_rows.append({'scenario': scenario, 'workload': workload, 'n': len(values), 'p50_ms': r7(values, .50), 'p95_r7_ms': r7(values, .95), 'p99_r7_ms': r7(values, .99), 'max_ms': max(values)})
latency_table = pl.DataFrame(latency_rows)
latency_table

In [ ]:
recovery_latency = []
for backlog, distributions in recovery['latency_summary_ns'].items():
    strict = distributions['strict_last_callback_to_polars_ns']
    recovery_latency.append({
        'replay_records': int(backlog),
        'n': strict['n'],
        'strict_last_p50_ms': strict['p50'] / 1e6,
        'strict_last_p95_r7_ms': strict['p95_r7'] / 1e6,
        'strict_last_p99_r7_ms': strict['p99_r7'] / 1e6,
        'recovery_duration_p50_ms': distributions['recovery_duration_ns']['p50'] / 1e6,
        'publication_to_polars_p50_ms': distributions['publication_to_polars_ns']['p50'] / 1e6,
    })
recovery_latency_table = pl.DataFrame(recovery_latency).sort('replay_records')
recovery_latency_table